# 🛒 Online Retail — Data Cleaning

Projeto de limpeza e padronização de dados de vendas utilizando **Python e Pandas**.
Foram realizadas análises de valores nulos, duplicados, inconsistências e tipos de dados.
Também foram tratados cancelamentos, valores negativos e informações inconsistentes.
Ao final, a base foi estruturada e preparada para futuras análises e visualizações.
**Tecnologias:** Python • Pandas • NumPy • Jupyter Notebook


In [1]:
import pandas as pd
import numpy as np

ARQUIVO_ENTRADA = 'online_retail_limpo.csv'

# O arquivo usa virgula como separador; os tipos sao definidos na leitura para evitar inferencias inconsistentes.
df_bruto = pd.read_csv(
    ARQUIVO_ENTRADA,
    sep=',',
    encoding='utf-8',
    low_memory=False,
    dtype={
        'InvoiceNo': 'string',
        'StockCode': 'string',
        'Description': 'string',
        'InvoiceDate': 'string',
        'Country': 'string'
    }
)

display(df_bruto.head(10))
display(df_bruto.info())
display(df_bruto.describe(include='all').T)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,25.50
1,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,15.30
2,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
3,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
5,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
6,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
7,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10
8,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,11.10
9,536367,21754,HOME BUILDING BLOCK WORD,3,2010-12-01 08:34:00,5.95,13047.0,United Kingdom,17.85


<class 'pandas.DataFrame'>
RangeIndex: 524878 entries, 0 to 524877
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    524878 non-null  string 
 1   StockCode    524878 non-null  string 
 2   Description  524878 non-null  string 
 3   Quantity     524878 non-null  int64  
 4   InvoiceDate  524878 non-null  string 
 5   UnitPrice    524878 non-null  float64
 6   CustomerID   392692 non-null  float64
 7   Country      524878 non-null  string 
 8   TotalPrice   524878 non-null  float64
dtypes: float64(3), int64(1), string(5)
memory usage: 71.0 MB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,524878,19960,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,524878,3922,85123A,2253,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,524878,4015,WHITE HANGING HEART T-LIGHT HOLDER,2311,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,524878.0,NaN,NaN,NaN,10.6166,156.280031,1.0,1.0,4.0,11.0,80995.0
InvoiceDate,524878,18499,2011-10-31 14:41:00,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,524878.0,NaN,NaN,NaN,3.922573,36.093028,0.001,1.25,2.08,4.13,13541.33
CustomerID,392692.0,NaN,NaN,NaN,15287.843865,1713.539549,12346.0,13955.0,15150.0,16791.0,18287.0
Country,524878,38,United Kingdom,479985,NaN,NaN,NaN,NaN,NaN,NaN,NaN
TotalPrice,524878.0,NaN,NaN,NaN,20.275399,271.693566,0.0,3.9,9.92,17.7,168469.6


In [7]:
display(list(df_limpo.columns))

['Numero_Nota_Fiscal',
 'Codigo_Produto',
 'Descricao_Produto',
 'Quantidade',
 'Data_Nota_Fiscal',
 'Preco_Unitario',
 'ID_Cliente',
 'Pais',
 'Preco_Total',
 'Cancelamento']

As colunas são: 
- InvoiceNo = Numero_Fatura
- StockCode = Codigo_Produto
- Description = Descricao_Produto
- Quantity = Quantidade
- InvoiceDate = Data_Fatura
- UnitPrice - Preco_Unitario
- CustomerID = ID_Cliente
- Country = Pais
- TotalPrice = Preco_Total

In [ ]:
df_limpo = df_bruto.copy()

# Padroniza campos textuais e converte datas e identificadores para tipos apropriados.
colunas_texto = ['InvoiceNo', 'StockCode', 'Description', 'Country']
for coluna in colunas_texto:
    df_limpo[coluna] = df_limpo[coluna].astype('string').str.strip()

df_limpo['InvoiceDate'] = pd.to_datetime(df_limpo['InvoiceDate'], errors='coerce')
df_limpo['Quantity'] = pd.to_numeric(df_limpo['Quantity'], errors='coerce')
df_limpo['UnitPrice'] = pd.to_numeric(df_limpo['UnitPrice'], errors='coerce')
df_limpo['CustomerID'] = pd.to_numeric(df_limpo['CustomerID'], errors='coerce').astype('Int64')
df_limpo['TotalPrice'] = pd.to_numeric(df_limpo['TotalPrice'], errors='coerce')

# Remove linhas sem dados essenciais, valores invalidos e duplicatas exatas.
colunas_obrigatorias = ['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate', 'Country']
df_limpo = df_limpo.dropna(subset=colunas_obrigatorias)
df_limpo = df_limpo[
    df_limpo['Quantity'].gt(0)
    & df_limpo['UnitPrice'].gt(0)
].copy()
df_limpo = df_limpo.drop_duplicates().reset_index(drop=True)

# Mantem cancelamentos identificados para que possam ser analisados separadamente.
df_limpo['Cancelamento'] = df_limpo['InvoiceNo'].str.upper().str.startswith('C')
df_limpo['TotalPrice'] = (df_limpo['Quantity'] * df_limpo['UnitPrice']).round(2)

display(df_limpo.info())
display(df_limpo.isna().sum().rename('valores_nulos').to_frame())

<class 'pandas.DataFrame'>
RangeIndex: 524878 entries, 0 to 524877
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   InvoiceNo     524878 non-null  string        
 1   StockCode     524878 non-null  string        
 2   Description   524878 non-null  string        
 3   Quantity      524878 non-null  int64         
 4   InvoiceDate   524878 non-null  datetime64[us]
 5   UnitPrice     524878 non-null  float64       
 6   CustomerID    392692 non-null  Int64         
 7   Country       524878 non-null  string        
 8   TotalPrice    524878 non-null  float64       
 9   Cancelamento  524878 non-null  boolean       
dtypes: Int64(1), boolean(1), datetime64[us](1), float64(2), int64(1), string(4)
memory usage: 63.3 MB


None

,valores_nulos
InvoiceNo,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,132186
Country,0
TotalPrice,0
Cancelamento,0


In [ ]:
df_limpo.rename(columns={
    'InvoiceNo': 'Numero_Nota_Fiscal',
    'StockCode': 'Codigo_Produto',
    'Description': 'Descricao_Produto',
    'Quantity': 'Quantidade',
    'InvoiceDate': 'Data_Nota_Fiscal',
    'UnitPrice': 'Preco_Unitario',
    'CustomerID': 'ID_Cliente',
    'Country': 'Pais',
    'TotalPrice': 'Preco_Total'
}, inplace=True)

ARQUIVO_SAIDA = 'online_retail_padronizado.csv'
df_limpo.to_csv(ARQUIVO_SAIDA, index=False, encoding='utf-8')

display(df_limpo.head())
display(df_limpo.dtypes.rename('tipo').to_frame())
print(f'Arquivo salvo: {ARQUIVO_SAIDA}')
print(f'Registros finais: {len(df_limpo):,}')
print(f'Cancelamentos: {int(df_limpo["Cancelamento"].sum()):,}')

,Numero_Nota_Fiscal,Codigo_Produto,Descricao_Produto,Quantidade,Data_Nota_Fiscal,Preco_Unitario,ID_Cliente,Pais,Preco_Total,Cancelamento
0,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,25.50,False
1,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,15.30,False
2,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False
3,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False
4,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,False


,tipo
Numero_Nota_Fiscal,string
Codigo_Produto,string
Descricao_Produto,string
Quantidade,int64
Data_Nota_Fiscal,datetime64[us]
Preco_Unitario,float64
ID_Cliente,Int64
Pais,string
Preco_Total,float64
Cancelamento,boolean


Arquivo salvo: online_retail_padronizado.csv
Registros finais: 524,878
Cancelamentos: 0
